In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
%cd /content/gdrive/MyDrive

Mounted at /content/gdrive
/content/gdrive/MyDrive


In [ ]:
# set env variables
TEST_DATASET = "SESAR_ZTC_test_multi_entire_filtered2.csv"
OUTPUT_FILE = "OUTPUT.json"

# Set Up

In [ ]:
!pip install datasets sentencepiece tokenizers bitsandbytes accelerate xformers einops
!pip install git+https://github.com/huggingface/transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 MB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.1/290.1 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.5/222.5 MB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 813.4 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━

# Download

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig

device = "cuda"
# This causes OOM

# model = AutoModelForCausalLM.from_pretrained(
#     "Open-Orca/Mistral-7B-OpenOrca").to(device)
# tokenizer = AutoTokenizer.from_pretrained(
#     "Open-Orca/Mistral-7B-OpenOrca")

In [ ]:
import transformers
model_id = "Open-Orca/Mistral-7B-OpenOrca"
bnb_config = transformers.BitsAndBytesConfig(
  load_in_4bit=True,
  bnb_4bit_use_double_quant=True,
  bnb_4bit_quant_type="nf4",
  bnb_4bit_compute_dtype=torch.bfloat16
)

model = transformers.AutoModelForCausalLM.from_pretrained(
  model_id,
  trust_remote_code=True,
  quantization_config=bnb_config,
  device_map='auto',
)

tokenizer = transformers.AutoTokenizer.from_pretrained(
  model_id,
)
model.config.use_cache = True

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/623 [00:00<?, ?B/s]

pytorch_model.bin.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

pytorch_model-00001-of-00002.bin:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/101 [00:00<?, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


# Inference

In [ ]:
inputs = tokenizer(
    "Orcas were not known to be drawn to mistral energy, but they were seen recently ",
    return_tensors="pt").to(device)
outputs = model.generate(
    **inputs, max_new_tokens=256, use_cache=True, do_sample=True,
    temperature=0.2, top_p=0.95)
text = tokenizer.batch_decode(outputs)[0]
print(text)

Setting `pad_token_id` to `eos_token_id`:32000 for open-end generation.


<s> Orcas were not known to be drawn to mistral energy, but they were seen recently 100 miles off the coast of Brazil.

The Brazilian Navy reported that a group of orcas were seen swimming around a platform in the Campos Basin, an area known for its oil production.

The orcas were seen swimming around the platform, which is 100 miles off the coast of Brazil, in the Campos Basin, an area known for its oil production.

The navy said the orcas were swimming around the platform, which is 100 miles off the coast of Brazil, in the Campos Basin, an area known for its oil production.

The navy said the orcas were swimming around the platform, which is 100 miles off the coast of Brazil, in the Campos Basin, an area known for its oil production.

The navy said the orcas were swimming around the platform, which is 100 miles off the coast of Brazil, in the Campos Basin, an area known for its oil production.

The navy said the orcas were swimming around the platform, which is 100 miles off the coas

In [ ]:
sys_prompt = "A chat."
prompt = "Tell me a joke."

prefix = "<|im_start|>"
suffix = "<|im_end|>\n"
sys_format = prefix + "system\n" + sys_prompt + suffix
user_format = prefix + "user\n" + prompt + suffix
assistant_format = prefix + "assistant\n"
input_text = sys_format + user_format + assistant_format

generation_config = GenerationConfig(
    max_length=256, temperature=1.1, top_p=0.95, repetition_penalty=1.0,
    do_sample=True, use_cache=True,
    eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.eos_token_id,
    transformers_version="4.34.0.dev0")

inputs = tokenizer(input_text, return_tensors="pt", return_attention_mask=True).to(device)
outputs = model.generate(**inputs, generation_config=generation_config)

text = tokenizer.batch_decode(outputs)[0]
print(text)

<s><|im_start|> system
A chat.<|im_end|><|im_start|> user
Tell me a joke.<|im_end|><|im_start|> assistant
 Two mice were talking, one had a tiny, tiny car. He said to his friend, "Hey, how does your new car run?" His friend replied, "Well, it's so tiny, it can run under any door!" The first mouse retorted, "Wow, that's really impressive, but mine runs in your pocket!"
<|im_end|>


In [ ]:
print(text.split("<|im_end|>")[2][len("<|im_start|> assistant"):].strip("\n"))

 Two mice were talking, one had a tiny, tiny car. He said to his friend, "Hey, how does your new car run?" His friend replied, "Well, it's so tiny, it can run under any door!" The first mouse retorted, "Wow, that's really impressive, but mine runs in your pocket!"


# Zero Shot

In [ ]:
# load test dataset
import pandas as pd
test_df = pd.read_csv(TEST_DATASET)

## Set Up - Read Taxonomy Files

In [ ]:
import json
leaf_material_types = list(open("unique_leaf_labels.txt").read().splitlines())
joined_leaf_material_types = "\n".join(leaf_material_types)

leaf_to_entire_path_mapping = {}
with open('leaf_to_parents_mapping.json') as f:
    leaf_to_entire_path_mapping = json.load(f)

mapping = json.load(open("label_to_parent_mapping.json"))
label_to_parent = {}
for k, v in mapping.items():
  if "material" in v:
    v.remove("material")
  label_to_parent[k] = v

child_to_parent = json.load(open("child_to_parent_mapping.json"))
def get_parent_labels(curr_labels):
  # return the parent labels (upper level labels)
  parent_labels = []
  for label in curr_labels:
    if label in child_to_parent and child_to_parent[label]:
      parent_labels.append(child_to_parent[label])
  parent_labels = list(set(parent_labels))
  return parent_labels


In [ ]:
import json

# mapping to description and individual fields that contain geology terms that need to be enriched
# this can be easily generated by using mapping of two columns in dataframe
desc_to_tax_map = json.load(open("description_to_taxonomy_train_all.json"))
desc_to_cm_map = json.load(open("description_to_collectionMethod_train_all.json"))
desc_to_desc_map = json.load(open("description_to_description_train_all.json"))

def trim_mapping(mapping):
  return {k.split("</s>")[0][len("<s>"):] : v for k,v in mapping.items()}
desc_to_tax_map = trim_mapping(desc_to_tax_map)
desc_to_cm_map = trim_mapping(desc_to_cm_map)
desc_to_desc_map = trim_mapping(desc_to_desc_map)

# Set Up - Prompt

In [ ]:
sys_prompt = "A chat."
prompt = "Tell me a joke."

prefix = "<|im_start|>"
suffix = "<|im_end|>\n"
sys_format = prefix + "system\n" + sys_prompt + suffix
user_format = prefix + "user\n" + prompt + suffix
assistant_format = prefix + "assistant\n"
input_text = sys_format + user_format + assistant_format

generation_config = GenerationConfig(
    max_length=256, temperature=1.1, top_p=0.95, repetition_penalty=1.0,
    do_sample=True, use_cache=True,
    eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.eos_token_id,
    transformers_version="4.34.0.dev0")


In [ ]:
def generate_answer(system_prompt, prompt):
  sys_format = prefix + "system\n" + sys_prompt + suffix
  user_format = prefix + "user\n" + prompt + suffix
  assistant_format = prefix + "assistant\n"
  input_text = sys_format + user_format + assistant_format

  generation_config = GenerationConfig(
    max_new_tokens=100, temperature=0.0001, top_p=0.95, repetition_penalty=1.0,
    do_sample=True, use_cache=True,
    eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.eos_token_id,
    transformers_version="4.34.0.dev0")

  inputs = tokenizer(input_text, return_tensors="pt", return_attention_mask=True).to(device)
  outputs = model.generate(**inputs, generation_config=generation_config)

  text = tokenizer.batch_decode(outputs)[0]
  text = text.split("<|im_end|>")[2][len("<|im_start|> assistant"):].strip("\n")
  print(text)

  return text

In [ ]:
# Summarize And Explain
def add_taxonomy_description(description, description_to_taxonomy):
  # add taxonomy description if exists and see if that helps
  if description in description_to_taxonomy and type(description_to_taxonomy[description]) == str:
    term = description_to_taxonomy[description]
    system_prompt = "You are a scientist. User will give you a geology term. You must generate a short description of the term."
    instruction = f"""You are a scientist. Your task is to give a brief one sentence description of material the geology term it consists.
    ###
    <<<
    Term:{term}
    >>>
    """
    return generate_answer(system_prompt, instruction).strip("\n")
  else:
    return None

def add_collectionMethod_description(description, description_to_cm):
  # add collectionMethod description if exists and see if that helps
  if description in  description_to_cm and type( description_to_cm[description]) == str:
    term = description_to_cm[description]
    system_prompt = "You are a scientist. User will give you a term that indicates how it collected a sample from nature. You must generate a short description of the term."
    instruction = f"""You are a scientist. Your task is to give a brief one sentence description of the given collection method of the sample.
    ###
    <<<
    Term:{term}
    >>>
    """
    return generate_answer(system_prompt, instruction).strip("\n")
  else:
    return None
def add_description_description(description, description_to_desc):
  # add collectionMethod description if exists and see if that helps
  if description in  description_to_desc and type( description_to_desc[description]) == str:
    term = description_to_desc[description]
    system_prompt = "You are a scientist. User will give you a term that indicates a description of the sample. You must generate a short explanation of that description."
    instruction = f"""You are a scientist. Your task is to give a one sentence explanation of the description of the sample.
    ###
    <<<
    Description:{term}
    >>>
    """
  else:
    return None


def generate_summary(sample_description):
  system_prompt = "You are a scientist. User will give you a description of a material sample it sampled from the nature. You must generate a summarized description of the sample."
  prompt = f"""You are a scientist. Your task is to give a brief one sentence summary of the given description, focusing on the parts that is helpful in determining the type of material that constitutes it.
  Include important fields in the summary such as the taxonomy informal classification, collection method, and values that determine the material type.
  ###
  <<<
  Description: {sample_description}
  >>>
  """

  return generate_answer(system_prompt, prompt)

def generate_summary_and_explanation(sample_description):
  summary = generate_summary(sample_description)

  taxonomy_rich_description = add_taxonomy_description(sample_description, desc_to_tax_map)
  if taxonomy_rich_description:
    #summary += enrich_field("fieldName", sample_description, taxonomy_rich_description.strip("\n"))
    summary += taxonomy_rich_description.strip("\n")
  collectionMethodDesc = add_collectionMethod_description(sample_description, desc_to_cm_map)
  if collectionMethodDesc:
    #summary += enrich_field("collectionMethod", sample_description, collectionMethodDesc.strip("\n"))
    summary += collectionMethodDesc.strip("\n")
  description_rich_description = add_description_description(sample_description, desc_to_desc_map)
  if description_rich_description:
    #summary += enrich_field("description", sample_description, description_rich_description.strip("\n"))
    summary += description_rich_description.strip("\n")
  print("Enriched summary : ", summary)
  return summary

In [ ]:
#CoT

def generate_reasoning(material_types, sample_description):
  system_prompt = "You are a scientist. User will give you a task. You must generate an answer to the task."
  instruction = f"""You are a scientist. Your task is to analyze the description of a material sample and determine the kind of material that constitutes it after <<<>>> into one of the predefined material types: \n
  {material_types} \n
  Let's think step by step.\n
  ###\n
  <<<
  Description: {sample_description}
  >>>
  """

  return generate_answer(system_prompt, instruction).strip("\n")

def generate_prediction_with_reasoning(material_types, sample_description, reasoning):
  system_prompt = "You are a scientist. User will give you a task. You must generate an answer to the task"
  instruction = f"""You are a scientist. Your task is to analyze the description of a material sample and determine the kind of material that constitutes it after <<<>>> into one of the predefined material types: \n
  {material_types} \n
  Let's think step by step.\n
  ###\n
  <<<
  Description: {sample_description}
  >>>
  {reasoning}
  Therefore, the answer(kind of material) is
  """
  return generate_answer(system_prompt, instruction).strip("\n")



In [ ]:
def generate_prediction(material_types, sample_description):
  system_prompt = "You are a scientist. User will give you a task. You must generate an answer to the task."
  instruction = f"""
  You are a scientist. Your task is to analyze the description of a material sample and determine the kind of material that constitutes it after <<<>>> into one of the predefined material types: \n
  {material_types} \n
  You will only respond with the material type. Do not include the word "Material type". Do not provide explanations or notes.
  <<<
  Description: {sample_description}
  Material type:
  >>>
  """
  return generate_answer(system_prompt, instruction).strip("\n")

In [ ]:
import nltk
nltk.download('punkt')
from nltk.tokenize import word_tokenize
from nltk.util import ngrams
from nltk.metrics import jaccard_distance

def jaccard_similarity(set1, set2):
    """
    Calculate Jaccard similarity between two sets.
    """
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return intersection / union

def tokenize_text(text):
    """
    Tokenize input text.
    """
    return set(word_tokenize(text.lower()))

def jaccard_highest_score(sample_text, set_of_texts):
    """
    Find the text with the highest Jaccard similarity score compared to the sample text.
    """
    sample_tokens = tokenize_text(sample_text)
    highest_score = 0
    most_similar_text = None
    print(set_of_texts)
    for text in set_of_texts:
        text_tokens = tokenize_text(text)
        similarity_score = jaccard_similarity(sample_tokens, text_tokens)

        if similarity_score > highest_score:
            highest_score = similarity_score
            most_similar_text = text

    return highest_score, most_similar_text # most similar material type

def extract_prediction(candidate_material_types, response):
  # do greedy text match
  result = []
  response = response.lower()
  if response.startswith("The material type is: "):
    response = response[len("The material type is: ")]
  for candidate_material_type in candidate_material_types:
    if candidate_material_type in response:
      result.append(candidate_material_type)
    # convert multi label type labels
    elif candidate_material_type == "rock or sediment":
      if "rock" in response.lower() or "sediment" in response.lower():
        result.append(candidate_material_type)
    elif candidate_material_type == "mixed soil sediment or rock" and "soil" in response.lower():
      result.append(candidate_material_type)
  if len(result) == 0:
    # use jaccard score
    higest_score, most_similar_material_types = jaccard_highest_score(response, candidate_material_types)
    #print(f"Jaccard score : {higest_score} : {most_similar_material_types}")
    if higest_score >= 0.5:
      result.append(most_similar_material_types)

  return result

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [ ]:
def zero_shot_prediction(test_df, summarize_and_explain = False, cot = False, OUTPUT_FILE="output.json"):
  test_prediction = []
  for idx, row in test_df.iterrows():
    sample_description = row['concatenated_text_B'].split("</s>")[0][len("<s>"):]
    input = sample_description

    # SummarizeExplain
    if summarize_and_explain:
      summary = generate_summary_and_explanation(sample_description)
      input = summary

    # ZTC-CoT
    if cot:
      reasoning = generate_reasoning(joined_leaf_material_types,input)
      output = generate_prediction_with_reasoning(joined_leaf_material_types, input, reasoning)
    else:
      output = generate_prediction(joined_leaf_material_types,input)

    prediction = extract_prediction(leaf_material_types, output)

    final_prediction = []
    if len(prediction) > 0:
      for pred in prediction:
        final_prediction.extend(leaf_to_entire_path_mapping[pred]) # get entire parents as well and add it to the prediction

    # TODO
    else:
      # level-up traversal recursively
      curr_labels = leaf_material_types
      parent_labels = get_parent_labels(curr_labels)

      while len(prediction) == 0 and len(parent_labels)>0:
        curr_labels = parent_labels
        joined_curr_labels = "\n".join(curr_labels)
        if cot:
          reasoning = generate_reasoning(joined_curr_labels,input)
          output = generate_prediction_with_reasoning(joined_curr_labels, input, reasoning)
        else:
          output = generate_prediction(joined_curr_labels,input)

        prediction = extract_prediction(curr_labels, output)
        print("Extracted labels: ", prediction)
        final_prediction = []
        for pred in prediction:
          final_prediction.append(pred)
          final_prediction.extend(label_to_parent[pred])

        # recurse up one level
        parent_labels = get_parent_labels(curr_labels)


    final_prediction = list(set(final_prediction))
    final_prediction = [x for x in final_prediction if x!= None and x!='material']
    print("Prediction : ", final_prediction)

    test_prediction.append(final_prediction)
    print("Gold: ",row["label_list"])
    print(f"{idx}-th prediction done\n\n")
    if idx % 100 == 0:
      with open(OUTPUT_FILE, 'w') as file:
        json.dump(test_prediction, file)

  with open(OUTPUT_FILE, 'w') as file:
    json.dump(test_prediction, file)

test_prediction = zero_shot_prediction(test_df, False, False, OUTPUT_FILE)

 Nephelinite
['quartz rich igneous rock', 'generic mudstone', 'quartz', 'slate', 'copper', 'hybrid sediment', 'plaster', 'mineral-phosphate, arsenate, or vanadate', 'glass rich igneous rock', 'iron rich sedimentary rock', 'brick clay', 'gold', 'pegmatite', 'limestone', 'trachytoid', 'massive sulphide', 'mud size sediment', 'gaseous material', 'mineral-organic compound', 'residual material', 'charcoal', 'greywacke', 'glass', 'paper', 'leather', 'doleritic rock', 'mineral-halide', 'carbonate sediment', 'tephritoid', 'chemical sedimentary material', 'non-aqueous liquid material', 'pyroclastic rock', 'plutonic igneous rock', 'iron rich sediment', 'foid gabbroid', 'mica', 'aphanite', 'bone', 'plant fiber', 'andesite', 'exotic composition igneous rock', 'hybrid sedimentary rock', 'shell', 'cinder', 'amber', 'foid dioritoid', 'frozen water', 'sand size sediment', 'dolomite', 'marble', 'porcelain', 'aplite', 'iron', 'travertine', 'mineral-borate', 'pumice', 'breccia gouge series', 'bucchero', 

KeyboardInterrupt: 

# Evaluate

In [ ]:
multi_to_label = {
    "rock or sediment": ["rock", "sediment"],
    "mixed soil sediment rock" : ["soil", "sediment", "rock"]
}

final_predicted_labels = test_prediction
for idx, labels in enumerate(test_prediction):
  for label in labels:
    if label in multi_to_label:
      labels.remove(label)
      labels.extend(multi_to_label[label])
      labels = list(set(labels))
  final_predicted_labels[idx] = labels # update

# assert
for idx, labels in enumerate(test_prediction):
  for label in labels:
    if label in multi_to_label:
      print("invalid")
      break
final_predicted_labels[-1]

In [ ]:
label_file="total_labels.txt" # file that stores all labels of iSamples taxonomy
gold_label_names = open(label_file).read().splitlines()

In [ ]:
true_labels = [x.split("/") for x in test_df['label_list'].tolist()]
true_labels

In [ ]:
## Multi label evaluation
from sklearn.metrics import classification_report
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
mlb.fit([gold_label_names])

true_labels_bin = mlb.transform(true_labels)
predicted_labels_bin = mlb.transform(final_predicted_labels)

print(classification_report(true_labels_bin, predicted_labels_bin, target_names=mlb.classes_))
report = classification_report(true_labels_bin, predicted_labels_bin, target_names=mlb.classes_, output_dict=True)

# Print the classification report
print(report)